In [1]:
from model import get_model

model = get_model(num_classes=10)

In [3]:
from dataset import get_dataloaders

train_loader, test_loader, class_names = get_dataloaders(
    "C:/Users/Tekin/Documents/Python/mil_vehicle_classifier/archive/dataset",
    batch_size=32,
    img_size=224
)

print(type(train_loader))
print(type(test_loader))
print(class_names)

Classes: ['Anti-aircraft', 'Armored combat support vehicles', 'Armored personnel carriers', 'Infantry fighting vehicles', 'Light armored vehicles', 'Mine-protected vehicles', 'Prime movers and trucks', 'Self-propelled artillery', 'light utility vehicles', 'tanks']
Train samples: 10414, Test samples: 3719
<class 'torch.utils.data.dataloader.DataLoader'>
<class 'torch.utils.data.dataloader.DataLoader'>
['Anti-aircraft', 'Armored combat support vehicles', 'Armored personnel carriers', 'Infantry fighting vehicles', 'Light armored vehicles', 'Mine-protected vehicles', 'Prime movers and trucks', 'Self-propelled artillery', 'light utility vehicles', 'tanks']


In [4]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(device)

cuda


In [5]:
print(train_loader)
print(type(train_loader))

<class 'torch.utils.data.dataloader.DataLoader'>


In [6]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)

True
12.1


In [7]:
# Unfreeze last 3 blocks
for param in model.features[-3:].parameters():
    param.requires_grad = True

# Recreate optimizer so it includes the newly unfrozen parameters
optimizer = torch.optim.Adam([
    {'params': model.features[-3:].parameters(), 'lr': 1e-5},  # unfrozen layers — small lr
    {'params': model.classifier.parameters(), 'lr': 1e-4}       # classifier — normal lr
])

criterion = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

num_epochs = 20
best_acc = 0.0

In [9]:

for param in model.features[-3:].parameters():
    param.requires_grad = True

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)


    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total

    scheduler.step()

    if epoch_acc > best_acc:
        best_acc = epoch_acc
        torch.save(model.state_dict(), "best_model.pth")

    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f} - Train Acc: {epoch_acc:.2f}%")



Epoch 1/20 - Loss: 1.8617 - Train Acc: 41.13%
Epoch 2/20 - Loss: 1.4885 - Train Acc: 54.06%
Epoch 3/20 - Loss: 1.2463 - Train Acc: 60.83%
Epoch 4/20 - Loss: 1.0669 - Train Acc: 66.43%
Epoch 5/20 - Loss: 0.9126 - Train Acc: 71.49%
Epoch 6/20 - Loss: 0.8236 - Train Acc: 74.63%
Epoch 7/20 - Loss: 0.7599 - Train Acc: 76.36%
Epoch 8/20 - Loss: 0.7206 - Train Acc: 77.50%
Epoch 9/20 - Loss: 0.6697 - Train Acc: 79.73%
Epoch 10/20 - Loss: 0.6317 - Train Acc: 80.62%
Epoch 11/20 - Loss: 0.5911 - Train Acc: 82.15%
Epoch 12/20 - Loss: 0.5705 - Train Acc: 83.21%
Epoch 13/20 - Loss: 0.5520 - Train Acc: 83.07%
Epoch 14/20 - Loss: 0.5437 - Train Acc: 83.32%
Epoch 15/20 - Loss: 0.5187 - Train Acc: 84.45%
Epoch 16/20 - Loss: 0.5086 - Train Acc: 84.66%
Epoch 17/20 - Loss: 0.5018 - Train Acc: 85.07%
Epoch 18/20 - Loss: 0.4904 - Train Acc: 85.51%
Epoch 19/20 - Loss: 0.4778 - Train Acc: 85.73%
Epoch 20/20 - Loss: 0.4734 - Train Acc: 85.97%


In [20]:
from PIL import Image
import torchvision.transforms as transforms
import torch

def predict(img_path, model, class_names):
    # Same transforms as your test set
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    img = Image.open(img_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)  # add batch dimension

    model.eval()
    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, pred_idx = torch.max(probabilities, 1)

    predicted_class = class_names[pred_idx.item()]
    confidence_pct = confidence.item() * 100

    print(f"Prediction: {predicted_class} ({confidence_pct:.1f}% confidence)")
    return predicted_class, confidence_pct

# Run it
predict(r"archive\dataset\validation\Light armored vehicles\Light armored vehicles_0_22.jpeg", model, class_names)

Prediction: Light armored vehicles (90.9% confidence)


('Light armored vehicles', 90.8780038356781)